**Age Model Architecture**

input: image

output: age

regression model

Dataset: UTKFace
Link to download:

part 1: https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669

part 2: https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426


part 3: https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395

### Get the Data

In [ ]:
#!rm -rf datasets/

In [1]:
# Download and unzip the datasets from UTKface

import tarfile
import urllib.request
import os

download_url_part1 = "https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669"
download_url_part2 = "https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426"
download_url_part3 = "https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395"

# Download the datasets
download_urls = [download_url_part1, download_url_part2, download_url_part3]
face_path = os.path.join("datasets", "faces")
os.makedirs(face_path, exist_ok=True)

for dataset_num in range(len(download_urls)):
  dataset_folder_name = "part" + str(dataset_num + 1)
  if (not os.path.exists(os.path.join(face_path, dataset_folder_name)) and not os.path.exists(os.path.join(face_path, "images"))):
    print(f"Extracting dataset {dataset_num+1}...")
    faces_tar_location = os.path.join(face_path, "faces" + str(dataset_num+1) + ".tgz")
    urllib.request.urlretrieve(download_urls[dataset_num], faces_tar_location)

    # extract tar dataset
    faces_tgz = tarfile.open(faces_tar_location)
    faces_tgz.extractall(path="datasets/faces")
    faces_tgz.close()

    # remove tar file
    os.remove(faces_tar_location)

  else:
    print(f"dataset {dataset_num+1} already exists")



dataset 1 already exists
dataset 2 already exists
dataset 3 already exists


In [2]:
# Combine all data into one folder
import shutil

faces_images_path = os.path.join(face_path, "images")
os.makedirs(faces_images_path, exist_ok=True)

# Get source folders to copy image data from
source_folders = []
for dataset in range(len(download_urls)):
  dataset_path = os.path.join("./datasets/faces/part" + str(dataset+1))
  if (os.path.exists(dataset_path)):
    source_folders.append(dataset_path)

for folder in source_folders:
  file_names = os.listdir(folder)
  for file_name in file_names:
    shutil.move(os.path.join(folder, file_name), faces_images_path)
  os.rmdir(folder) # remove directory because we don't need it anymore

In [3]:
!pip3 install pillow rich rich-pixels
!pip3 install setuptools==81.0
!pip3 install face-recognition
!pip3 install git+https://github.com/ageitgey/face_recognition_models

  Cloning https://github.com/ageitgey/face_recognition_models to /tmp/pip-req-build-a55a8hw5
  Running command git clone --filter=blob:none --quiet https://github.com/ageitgey/face_recognition_models /tmp/pip-req-build-a55a8hw5
  Resolved https://github.com/ageitgey/face_recognition_models to commit e67de717267507d1e9246de95692eb8be736ab61
  Preparing metadata (setup.py) ... done


In [4]:
file_names = os.listdir(faces_images_path)
print(f"Number of images: {len(file_names)}")

Number of images: 24106


### Cropping Images & Adding Padding

In [5]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from face_processor import process_single_image

os.makedirs(os.path.join(face_path, "cropped"), exist_ok=True)
test_set = file_names # test batch

results = []
futures = []
num_workers = 30
images_already_processed = os.listdir("./datasets/faces/cropped")

with ProcessPoolExecutor(max_workers=num_workers) as ppe:
  for img in test_set:
    if img not in images_already_processed:
      submission = ppe.submit(process_single_image, img)
      futures.append(submission)
  for fut in as_completed(futures):
    results.append(fut.result())

num_success = 0
for fn, success_boolean, msg in results:
  if success_boolean:
    num_success += 1

print(f"Processed {num_success}/{len(test_set)} images successfully")

/home/mklema/ml_group_project/.venv/lib/python3.11/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


1_0_2_20161219162850150.jpg
3_1_2_20161219162324150.jpg
16_0_0_20170110232445820.jpg
30_0_2_20170116163429712.jpg
45_0_1_20170117135648741.jpg
2_1_2_20161219200416275.jpg
84_1_0_20170120225434057.jpg
14_1_0_20170109214714069.jpg
9_1_3_20161220220026219.jpg
1_0_3_20161220142927974.jpg
1_0_4_20170103210247922.jpg
1_1_0_20161219154808116.jpg
75_0_0_20170111205746197.jpg
90_0_0_20170111205717074.jpg
43_0_1_20170117175541909.jpg
26_0_3_20170117153104987.jpg
110_1_1_20170110155117522.jpg
2_1_2_20161219161905006.jpg
60_1_1_20170120221732551.jpg
1_1_2_20161219203436580.jpg
77_0_0_20170113210319811.jpg
45_0_1_20170117165650390.jpg
56_0_0_20170105164901731.jpg
1_0_2_20161219222722535.jpg
1_0_2_20161219154727981.jpg
8_0_2_20161219193349427.jpg
1_1_0_20161219211652253.jpg
45_0_1_20170113152719963.jpg
1_1_3_20161220143154439.jpg
32_0_1_20170103182548641.jpg
2_0_2_20161219190604355.jpg
1_1_2_20161219141436353.jpg
29_1_1_20170114024736192.jpg
3_1_2_20161219210417277.jpg
86_0_0_20170111205938881.jpg
2

In [6]:
import random

# Shuffling image data before splitting it

random.seed(49328042)
images_shuffled = os.listdir("./datasets/faces/cropped")
random.shuffle(images_shuffled)

age_labels = []

for file in images_shuffled:
    age_label = int(file[0:file.index('_')])
    age_labels.append(age_label)


In [7]:
# Split the data into training, validation, and testing sets

train_data = images_shuffled[:16653]
train_labels = age_labels[:16653]

validation_data = images_shuffled[16653:19053]
validation_labels = age_labels[16653:19053]

testing_data = images_shuffled[19053:]
testing_labels = age_labels[19053:]


In [8]:
print(len(age_labels))

23623


In [9]:
# Create image set directories based on labels

import os, shutil, pathlib

original_dir = "./" + str(pathlib.Path("datasets/faces/cropped"))
base_training_dir = "./" + str(pathlib.Path("datasets/faces/training"))

def make_data_sets(subdir, starting_index, ending_index):
    for img_index in range(starting_index, ending_index):
        dir_path = f"{base_training_dir}/{subdir}/{str(age_labels[img_index])}"
        os.makedirs(dir_path, exist_ok=True)

        image_path = os.path.join(original_dir, images_shuffled[img_index])
        if (image_path not in os.listdir(dir_path)):
            shutil.move(image_path, dir_path)

if (len(os.listdir(original_dir)) > 0):
    make_data_sets("train", 0, 16650)
    make_data_sets("validation", 16650, 19050)
    make_data_sets("test", 19050, 23623)



In [10]:
# set max thread count for model training

import os

os.environ["OMP_NUM_THREADS"] = "26"
os.environ["TF_NUM_INTRAOP_THREADS"] = "26"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

from tensorflow.config.threading import set_intra_op_parallelism_threads
set_intra_op_parallelism_threads(26)

I0000 00:00:1773216181.840059    5588 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773216183.426596    5588 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773216189.432727    5588 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [11]:
# Create datasets based on directories
from tensorflow.keras.utils import image_dataset_from_directory

batch_size = 32

# 16650 files
train_dataset = image_dataset_from_directory(
    f"{base_training_dir}/train",
    image_size=(256,256),
    batch_size=batch_size,
)

# 2400 files
validation_dataset = image_dataset_from_directory(
    f"{base_training_dir}/validation",
    image_size=(256,256),
    batch_size=batch_size
)

# 5000 files
test_dataset = image_dataset_from_directory(
    f"{base_training_dir}/test",
    image_size=(256,256),
    batch_size=batch_size,
)

Found 16650 files belonging to 102 classes.


E0000 00:00:1773216197.520849    5588 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 2400 files belonging to 95 classes.
Found 4573 files belonging to 100 classes.


In [12]:
for data_batch, labels_batch in train_dataset:
    print(f"Shape of data set: {data_batch.shape}")
    print(f"Shape of label set: {labels_batch.shape}")
    break

Shape of data set: (32, 256, 256, 3)
Shape of label set: (32,)


### Building the CNN

In [13]:
%pip install tensorflow
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential

# 1 Input Layer
# 3 Convolutional Layers
# 3 Max Pooling Layers
# 2 Dense Layers (after flattening)

cnn_model = Sequential([
    layers.Input(shape=(256,256,3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(256, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1)
])

cnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │     1,605,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,994,113 (7.61 MB)

 Trainable params: 1,994,113 (7.61 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable

# Custom metric to track percent of age predictions correct within N-year tolerance.
@register_keras_serializable(package="Custom")
class WithinNYears(tf.keras.metrics.Metric):
    def __init__(self, tolerance=None, dtype=None, name=None, **kwargs):
        # Backward-compatible load path: if older saved configs omit tolerance
        if tolerance is None and isinstance(name, str):
            match = re.match(r"within_(\d+)_years", name)
            if match:
                tolerance = int(match.group(1))

        if tolerance is None:
            tolerance = 2

        super().__init__(name=name or f"within_{tolerance}_years", dtype=dtype, **kwargs)
        self.tolerance = int(tolerance)
        self.correct = self.add_weight(name="correct", initializer="zeros")
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.squeeze(y_pred, axis=-1) # get last tensor (output) and transform (batch_size,1) to (batch_size,)
        y_true = tf.cast(y_true, tf.float32) 
        within = tf.abs(y_true - y_pred) <= self.tolerance
        self.correct.assign_add(tf.reduce_sum(tf.cast(within, tf.float32))) # convert booleans to floats and get total sum and add to correct
        self.total.assign_add(tf.cast(tf.size(y_true), tf.float32)) # total is just size of y_true

    def result(self):
        return self.correct / self.total
    
    def reset(self): # runs once per epoch
        self.correct.assign(0)
        self.total.assign(0)

    def get_config(self):
        config = super().get_config()
        config.update({"tolerance": self.tolerance})
        return config

In [16]:
# compiling model and adding callbacks

cnn_model.compile(
    loss=tf.keras.losses.Huber(delta=3.0),
    optimizer=tf.keras.optimizers.AdamW(),
    metrics=["mae", WithinNYears(2), WithinNYears(5), WithinNYears(10)]
)

# saves best model
# stops if no improvement after 6 epochs

callbacks = [
    keras.callbacks.ModelCheckpoint( 
        filepath="cnn_model_best.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
]



In [2]:
training_history = cnn_model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks
)

NameError: name 'cnn_model' is not defined

### Tuning Model

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    mae = history.history["mae"]
    val_mae = history.history["val_mae"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(mae) + 1)

    plt.plot(epochs, mae, "bo", label="Training Mae")
    plt.plot(epochs, val_mae, "b", label="Validation Mae")
    plt.title("Training and Validation MAE")
    plt.xlabel("Epochs")
    plt.ylabel("MAE")
    plt.legend()
    plt.figure()

    plt.plot(epochs[1:], loss[1:], "bo", label="Training Loss")
    plt.plot(epochs[1:], val_loss[1:], "b", label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


In [ ]:
plot_training_history(training_history)

### Evaluating the Model

In [ ]:
custom_objects = {"WithinNYears": WithinNYears}
test_model = keras.models.load_model("cnn_model_best.keras", custom_objects=custom_objects)
test_loss = test_model.evaluate(test_dataset)
print(f"Test Loss: {test_loss[0]:.3f}\nTest MAE: {test_loss[2]:.3f}\nTest within 10y: {test_loss[5]:.3f}\nTest within 5y: {test_loss[4]:.3f}\nTest within 2y: {test_loss[3]:.3f}")

In [ ]:
# from PIL import Image, ImageOps
# import face_recognition
# import os

# !pwd

# image = face_recognition.load_image_file(os.path.join("./", 'lisa.jpg'))
# face_locations = face_recognition.face_locations(image, model="hog")

# if len(face_locations) > 0: # face found in image

#     # Get Cropped Image
#     top, right, bottom, left = face_locations[0]
#     face_image = image[top:bottom, left:right]
#     pil_image = Image.fromarray(face_image)

#     # add padding
#     processed_image = ImageOps.pad(
#         pil_image,
#         (160,160),
#         Image.Resampling.LANCZOS,
#         (0,0,0) # black padding
#     )

#     processed_image.save(f"cropped_lisa.jpg")

In [ ]:
# import numpy as np
# from PIL import Image
# from tensorflow import keras
# import os

# img_path = "cropped_lisa.jpg"
# model_path = "cnn_model_best.keras"

# if not os.path.exists(img_path):
#     raise FileNotFoundError(f"Could not find {img_path}. Run the crop cell first.")

# if "test_model" not in globals():
#     if not os.path.exists(model_path):
#         raise FileNotFoundError(f"Could not find {model_path}.")
#     test_model = keras.models.load_model(model_path, compile=False)

# img = Image.open(img_path)
# img_array = np.array(img, dtype=np.float32)
# img_batch = np.expand_dims(img_array, axis=0)

# print(f"Input tensor shape: {img_batch.shape}")

# pred = test_model.predict(img_batch, verbose=0)
# predicted_age = float(pred[0][0])
# print(f"Predicted age: {predicted_age:.2f}")